In [7]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt  # for making figures
%matplotlib inline

In [8]:
words = open('names.txt', 'r').read().splitlines()

In [9]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)

In [10]:
block_size = 3 

def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size 
        for ch in w + '.': 
            ix = stoi[ch]  
            X.append(context) 
            Y.append(ix)
            context = context[1:] + [ix]  
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

Xtr, Ytr = build_dataset(words[:n1])    # 训练集
Xdev, Ydev = build_dataset(words[n1:n2])# 验证集
Xte, Yte = build_dataset(words[n2:])    # 测试集

In [11]:
n_embd = 10 
n_hidden = 200 

g = torch.Generator().manual_seed(2147483647) 
C  = torch.randn((vocab_size, n_embd),            generator=g)
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)* (5/3)/(n_embd * block_size)**0.5
W2 = torch.randn((n_hidden, vocab_size),          generator=g)* 0.001 
b2 = torch.randn(vocab_size,                      generator=g)* 0 

bngain = torch.ones((1,n_hidden))
bnbias = torch.zeros((1,n_hidden))

bnmean_running = torch.zeros((1,n_hidden))
bnstd_running = torch.ones((1,n_hidden))

parameters = [C, W1, W2, b2,bngain,bnbias] 
print(sum(p.nelement() for p in parameters)) 
for p in parameters:
  p.requires_grad = True 

12097


In [ ]:
max_steps = 200000 
batch_size = 32    
lossi = []

for i in range(max_steps):

    ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
    Xb, Yb = Xtr[ix], Ytr[ix] 

    # 前向传播
    emb = C[Xb] # 嵌入操作得到样本的稠密向量/词向量
    embcat = emb.view(emb.shape[0], -1) # 将上下文向量拼接作为输入内容
    hpreact = embcat @ W1 #+ b1 # 隐藏层神经元计算
    
    bnmeani = hpreact.mean(0, keepdim=True)
    bnstdi = hpreact.std(0, keepdim=True)
    hpreact = bngain * (hpreact - bnmeani ) /  bnstdi + bnbias

    with torch.no_grad():
        bnmean_running=bnmean_running*0.999+bnmeani*0.001
        bnstd_running=bnstd_running*0.999+bnstdi*0.001
    
    h = torch.tanh(hpreact) # 隐藏层神经元非线性输出
    logits = h @ W2 + b2 # 输出层神经元计算
    loss = F.cross_entropy(logits, Yb) # 损失函数 负对数似然均值

    # 反向传播
    for p in parameters:
        p.grad = None
    loss.backward()

    # 更新梯度
    lr = 0.01 if i < 100000 else 0.001
    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0:
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
    lossi.append(loss.log10().item())

      0/ 200000: 2.2337
  10000/ 200000: 1.9732
  20000/ 200000: 2.2077
  30000/ 200000: 2.3755
  40000/ 200000: 2.0676
  50000/ 200000: 2.1705
  60000/ 200000: 2.1255
  70000/ 200000: 2.4859


In [65]:
# plt.hist(h.view([6400]).tolist(),50);

In [64]:
# plt.hist(hpreact.view([6400]).tolist(),50);

In [63]:
# plt.plot(lossi)

In [13]:
with torch.no_grad():
    emb = C[Xtr]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1
    bnmean = hpreact.mean(0, keepdim=True)
    bnstd = hpreact.std(0, keepdim=True)

In [26]:
@torch.no_grad() # 修饰函数告诉pytorch底层不需要去维护计算图，因为我们不需要反向传播
def split_loss(split):# 分别计算不同数据集的损失，只计算损失值不进行迭代所以不需要反向传播的过程
    x,y = {
        'train': (Xtr, Ytr),
        'val': (Xdev, Ydev),
        'test': (Xte, Yte),
    }[split]
    emb = C[x] # (N, block_size, n_embd)
    embcat = emb.view(emb.shape[0], -1) # concat into (N, block_size * n_embd)
    hpreact = embcat @ W1 
    hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
    #hpreact = bngain * (hpreact - bnmean) / bnstd + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2 
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

train 2.0507616996765137
val 2.099439859390259


In [27]:
g = torch.Generator().manual_seed(2147483647 + 10)
for _ in range(20):
    out = []
    context = [0] * block_size  
    while True:
        emb = C[torch.tensor([context])] 
        embcat = emb.view(emb.shape[0], -1) 
        hpreact = embcat @ W1 
        hpreact = bngain * (hpreact - bnmean_running) / bnstd_running + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out))

carman.
amelle.
khi.
mili.
thil.
skanden.
jazonel.
deliah.
jareei.
ner.
kentzie.
ivia.
leggy.
ham.
joce.
quintis.
lilea.
jamique.
jeron.
jarisi.
